# B-15: Direct Rollout-Based Hyperparameter Tuning

**Objective:** Test whether scoring hyperparameter combos by their actual 365-day rollout performance
(instead of one-step CV, D-58/B-14's proxy) finds better hyperparameters than either B-10's hand-tuned
baseline (D-54) or B-14's CV-tuned configs (D-58).

**Design:**
- **Stage 1 (this notebook, executed inline):** manual parameter grid (RF 9 / XGB 12 / LightGBM 12 = 33
  combos) scored by rollout R2 at a single anchor (2021-12-16), shortlisted top-3 per model, stability-
  checked at a second, differently-behaved anchor (2019-12-16, per D-53's addendum), winner = **combined
  rank** (mean of n-weighted R2 across both anchors).
- **Stage 2 (script extension `b15_multi_anchor.py`):** winners plugged into the exact same 5-anchor
  (2018-2022) rollout mechanism as B-10/B-14, pooled training (T2/T4/T9), Tower 4 evaluation, 4-model
  ensemble (RF+XGB+LightGBM+SARIMAX) -- fixing B-14's 3-model-only composition mismatch.
- **Stage 3 (script extension `compile_b15_results.py`):** 3-way comparison vs B-10/B-14, aligned by
  normalized model family (raw model names differ across the three sources).
- **Addendum (script extensions `b15_cross_tower_eval.py`/`b15_t9_rollout_grid_search.py`/
  `b15_t9_multi_anchor.py`):** do T4-tuned hyperparameters generalize to Tower 2/9? Does tuning
  separately for Tower 9 do any better?

Per this project's established norm (B-09's own "don't trust a single anchor" lesson, D-53), Stage 1 is
the smoke test executed here; Stages 2/3 and the addendum are multi-anchor sweeps that run as separate
scripts (same precedent as every B-09-B14 notebook) -- their results are summarized below with the
scripts referenced, not re-run inline, since a single 5-anchor sweep already takes several minutes.

Full narrative and final numbers: `b15_results.md`.

In [1]:
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")
import models.recursive_rollout as rr

HOURLY = Path("../../data/Hourly")
RESULTS = Path("../../results")

N_DAYS = 365
TOWERS = [2, 4, 9]
TOWER_MAIN = 4
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]

# Grid definitions (33 total: 9 RF + 12 XGB + 12 LGB) -- bracketing B-10/B-14's established values
RF_GRID = {"max_features": [0.3, 0.5, 0.7], "min_samples_leaf": [10, 20, 50]}
XGB_GRID = {"max_depth": [2, 3], "learning_rate": [0.01, 0.02], "min_child_weight": [5, 10, 20]}
LGB_GRID = {"num_leaves": [7, 15], "min_child_samples": [10, 20, 50], "learning_rate": [0.02, 0.05]}

## Stage 1a: Data loading + pooled training setup, per anchor

In [2]:
dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in TOWERS}
feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM

def run_grid_search_at_anchor(anchor_year):
    # Fit all 33 combos at a single anchor, scored by rollout R2 against Tower 4. Returns the
    # full per-combo bin_metrics results plus the top-3-per-model shortlist (by 2021 rank).
    anchor = pd.Timestamp(f"{anchor_year}-12-16")
    target_dates = pd.date_range(anchor + pd.Timedelta(days=1), periods=N_DAYS, freq="D")

    pool = []
    for t in TOWERS:
        df = T[t].copy()
        df["target"] = df["y_gapfilled"]
        for d in DUM:
            df[d] = 1.0 if d == f"is_t{t}" else 0.0
        pool.append(df[df.index <= anchor])
    tr = pd.concat(pool)
    tr = tr[tr["target"].notna()]

    df4 = T[TOWER_MAIN]
    history_init = df4.loc[:anchor, "y_gapfilled"].copy()
    fx_frame = df4.loc[target_dates, FX_B + ["ar_fc_dlag1"]].copy()
    fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0
    y_true_full = pd.Series(df4.loc[target_dates, "y_observed"].values, index=target_dates)
    anchor_val = df4.loc[anchor, "y_gapfilled"]
    persist = rr.chain_persistence(anchor_val, N_DAYS)

    imp = SimpleImputer(strategy="mean")
    Xi = imp.fit_transform(tr[feat_cols].values)

    results, rf_combos, xgb_combos, lgb_combos = [], [], [], []

    for mf in RF_GRID["max_features"]:
        for msl in RF_GRID["min_samples_leaf"]:
            rf = RandomForestRegressor(n_estimators=500, max_features=mf, min_samples_leaf=msl, n_jobs=-1, random_state=42)
            rf.fit(Xi, tr["target"].values)
            chain = rr.tree_rollout(rf, imp, feat_cols, fx_frame, history_init, anchor, n_days=N_DAYS)
            bm = rr.bin_metrics(y_true_full.values, chain.reindex(target_dates).values, target_dates, anchor, y_persist=persist)
            bm["model"], bm["max_features"], bm["min_samples_leaf"], bm["anchor_year"] = "RF", mf, msl, anchor_year
            results.append(bm)
            rf_combos.append((mf, msl, (bm["R2"] * bm["n"]).sum() / bm["n"].sum()))

    for md_ in XGB_GRID["max_depth"]:
        for lr in XGB_GRID["learning_rate"]:
            for mcw in XGB_GRID["min_child_weight"]:
                xgb = XGBRegressor(max_depth=md_, learning_rate=lr, min_child_weight=mcw, n_estimators=400,
                                    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
                xgb.fit(Xi, tr["target"].values)
                chain = rr.tree_rollout(xgb, imp, feat_cols, fx_frame, history_init, anchor, n_days=N_DAYS)
                bm = rr.bin_metrics(y_true_full.values, chain.reindex(target_dates).values, target_dates, anchor, y_persist=persist)
                bm["model"], bm["max_depth"], bm["learning_rate"], bm["min_child_weight"], bm["anchor_year"] = "XGB", md_, lr, mcw, anchor_year
                results.append(bm)
                xgb_combos.append((md_, lr, mcw, (bm["R2"] * bm["n"]).sum() / bm["n"].sum()))

    for nl in LGB_GRID["num_leaves"]:
        for mcs in LGB_GRID["min_child_samples"]:
            for lr in LGB_GRID["learning_rate"]:
                lgb = LGBMRegressor(num_leaves=nl, min_child_samples=mcs, learning_rate=lr, n_estimators=400,
                                     subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbosity=-1)
                lgb.fit(Xi, tr["target"].values)
                chain = rr.tree_rollout(lgb, imp, feat_cols, fx_frame, history_init, anchor, n_days=N_DAYS)
                bm = rr.bin_metrics(y_true_full.values, chain.reindex(target_dates).values, target_dates, anchor, y_persist=persist)
                bm["model"], bm["num_leaves"], bm["min_child_samples"], bm["learning_rate"], bm["anchor_year"] = "LGB", nl, mcs, lr, anchor_year
                results.append(bm)
                lgb_combos.append((nl, mcs, lr, (bm["R2"] * bm["n"]).sum() / bm["n"].sum()))

    R = pd.concat(results, ignore_index=True)
    rf_combos.sort(key=lambda x: x[2], reverse=True)
    xgb_combos.sort(key=lambda x: x[3], reverse=True)
    lgb_combos.sort(key=lambda x: x[3], reverse=True)
    shortlist = {"RF": rf_combos[:3], "XGB": xgb_combos[:3], "LGB": lgb_combos[:3]}
    return R, shortlist

print("Setup complete: 33-combo grid (RF 9 / XGB 12 / LightGBM 12), Tower 4 evaluation.")

Setup complete: 33-combo grid (RF 9 / XGB 12 / LightGBM 12), Tower 4 evaluation.


## Stage 1b: search anchor (2021-12-16) -- all 33 combos

In [3]:
search_results, shortlist = run_grid_search_at_anchor(2021)
search_results.to_csv(RESULTS/"b15_rollout_grid_search.csv", index=False)
print(f"Search complete: {len(search_results)} rows saved to b15_rollout_grid_search.csv")
print("\nTop-3 shortlist per model (by 2021 rank):")
for m, combos in shortlist.items():
    print(f"  {m}: {combos}")

Search complete: 198 rows saved to b15_rollout_grid_search.csv

Top-3 shortlist per model (by 2021 rank):
  RF: [(0.7, 10, np.float64(-0.10718768328445748)), (0.7, 50, np.float64(-0.10789149560117302)), (0.3, 20, np.float64(-0.11327565982404694))]
  XGB: [(3, 0.01, 20, np.float64(-0.0016011730205278574)), (3, 0.01, 5, np.float64(-0.00843401759530791)), (3, 0.01, 10, np.float64(-0.015360703812316724))]
  LGB: [(7, 10, 0.02, np.float64(-0.043390029325513196)), (7, 20, 0.02, np.float64(-0.07656891495601174)), (7, 20, 0.05, np.float64(-0.08285630498533723))]


## Stage 1c: stability check (2019-12-16) + combined-rank winner selection

2019 is a differently-behaved anchor (D-53's addendum: 2019/2020 lack the late-window degradation
2018/2021 show) -- re-scoring the top-3 shortlisted combos per model here, then picking the winner by
the **combined** (mean) rank across both anchors, guards against over-fitting the winner choice to
2021's specific idiosyncrasies (this project's own repeated "don't trust a single anchor" lesson,
applied here to the tuning method itself, not just to reporting results).

In [4]:
stability_results, _ = run_grid_search_at_anchor(2019)

rf_params, xgb_params, lgb_params = shortlist["RF"], shortlist["XGB"], shortlist["LGB"]
stability_filtered = pd.DataFrame()
for mf, msl, _ in rf_params:
    mask = (stability_results["model"] == "RF") & (stability_results["max_features"] == mf) & (stability_results["min_samples_leaf"] == msl)
    stability_filtered = pd.concat([stability_filtered, stability_results[mask]])
for md_, lr, mcw, _ in xgb_params:
    mask = (stability_results["model"] == "XGB") & (stability_results["max_depth"] == md_) & (stability_results["learning_rate"] == lr) & (stability_results["min_child_weight"] == mcw)
    stability_filtered = pd.concat([stability_filtered, stability_results[mask]])
for nl, mcs, lr, _ in lgb_params:
    mask = (stability_results["model"] == "LGB") & (stability_results["num_leaves"] == nl) & (stability_results["min_child_samples"] == mcs) & (stability_results["learning_rate"] == lr)
    stability_filtered = pd.concat([stability_filtered, stability_results[mask]])
stability_filtered.to_csv(RESULTS/"b15_stability_check.csv", index=False)
print(f"Stability check complete: {len(stability_filtered)} rows saved to b15_stability_check.csv")

Stability check complete: 54 rows saved to b15_stability_check.csv


In [5]:
def wavg_r2(mask):
    sub = stability_results[mask]
    return (sub["R2"] * sub["n"]).sum() / sub["n"].sum() if sub["n"].sum() > 0 else np.nan

winners = {}

print("RF:")
rf_scored = []
for mf, msl, r2_2021 in rf_params:
    mask = (stability_results["model"] == "RF") & (stability_results["max_features"] == mf) & (stability_results["min_samples_leaf"] == msl)
    r2_2019 = wavg_r2(mask)
    avg = np.nanmean([r2_2021, r2_2019])
    rf_scored.append((mf, msl, avg))
    print(f"  max_features={mf} min_samples_leaf={msl}: 2021={r2_2021:.4f} 2019={r2_2019:.4f} avg={avg:.4f}")
best_rf = max(rf_scored, key=lambda x: x[2])
winners["RF"] = {"max_features": best_rf[0], "min_samples_leaf": best_rf[1], "chosen_by": "combined_2021_2019"}
print(f"  WINNER: max_features={best_rf[0]} min_samples_leaf={best_rf[1]}\n")

print("XGB:")
xgb_scored = []
for md_, lr, mcw, r2_2021 in xgb_params:
    mask = (stability_results["model"] == "XGB") & (stability_results["max_depth"] == md_) & (stability_results["learning_rate"] == lr) & (stability_results["min_child_weight"] == mcw)
    r2_2019 = wavg_r2(mask)
    avg = np.nanmean([r2_2021, r2_2019])
    xgb_scored.append((md_, lr, mcw, avg))
    print(f"  max_depth={md_} lr={lr} min_child_weight={mcw}: 2021={r2_2021:.4f} 2019={r2_2019:.4f} avg={avg:.4f}")
best_xgb = max(xgb_scored, key=lambda x: x[3])
winners["XGB"] = {"max_depth": best_xgb[0], "learning_rate": best_xgb[1], "min_child_weight": best_xgb[2], "chosen_by": "combined_2021_2019"}
print(f"  WINNER: max_depth={best_xgb[0]} learning_rate={best_xgb[1]} min_child_weight={best_xgb[2]}\n")

print("LightGBM:")
lgb_scored = []
for nl, mcs, lr, r2_2021 in lgb_params:
    mask = (stability_results["model"] == "LGB") & (stability_results["num_leaves"] == nl) & (stability_results["min_child_samples"] == mcs) & (stability_results["learning_rate"] == lr)
    r2_2019 = wavg_r2(mask)
    avg = np.nanmean([r2_2021, r2_2019])
    lgb_scored.append((nl, mcs, lr, avg))
    print(f"  num_leaves={nl} min_child_samples={mcs} lr={lr}: 2021={r2_2021:.4f} 2019={r2_2019:.4f} avg={avg:.4f}")
best_lgb = max(lgb_scored, key=lambda x: x[3])
winners["LGB"] = {"num_leaves": best_lgb[0], "min_child_samples": best_lgb[1], "learning_rate": best_lgb[2], "chosen_by": "combined_2021_2019"}
print(f"  WINNER: num_leaves={best_lgb[0]} min_child_samples={best_lgb[1]} learning_rate={best_lgb[2]}\n")

winners_df = pd.DataFrame([
    {"model": "RF", **winners["RF"]},
    {"model": "XGB", **winners["XGB"]},
    {"model": "LGB", **winners["LGB"]},
])
winners_df.to_csv(RESULTS/"b15_winners.csv", index=False)
print("Saved winners to b15_winners.csv:")
winners_df

RF:
  max_features=0.7 min_samples_leaf=10: 2021=-0.1072 2019=-0.0107 avg=-0.0590
  max_features=0.7 min_samples_leaf=50: 2021=-0.1079 2019=0.0217 avg=-0.0431
  max_features=0.3 min_samples_leaf=20: 2021=-0.1133 2019=0.0335 avg=-0.0399
  WINNER: max_features=0.3 min_samples_leaf=20

XGB:
  max_depth=3 lr=0.01 min_child_weight=20: 2021=-0.0016 2019=0.0595 avg=0.0290
  max_depth=3 lr=0.01 min_child_weight=5: 2021=-0.0084 2019=0.0448 avg=0.0182
  max_depth=3 lr=0.01 min_child_weight=10: 2021=-0.0154 2019=0.0518 avg=0.0182
  WINNER: max_depth=3 learning_rate=0.01 min_child_weight=20

LightGBM:
  num_leaves=7 min_child_samples=10 lr=0.02: 2021=-0.0434 2019=0.1000 avg=0.0283
  num_leaves=7 min_child_samples=20 lr=0.02: 2021=-0.0766 2019=0.1104 avg=0.0169
  num_leaves=7 min_child_samples=20 lr=0.05: 2021=-0.0829 2019=0.1722 avg=0.0447
  WINNER: num_leaves=7 min_child_samples=20 learning_rate=0.05

Saved winners to b15_winners.csv:


,model,max_features,min_samples_leaf,chosen_by,max_depth,learning_rate,min_child_weight,num_leaves,min_child_samples
0,RF,0.3,20.0,combined_2021_2019,NaN,NaN,NaN,NaN,NaN
1,XGB,NaN,NaN,combined_2021_2019,3.0,0.01,20.0,NaN,NaN
2,LGB,NaN,NaN,combined_2021_2019,NaN,0.05,NaN,7.0,20.0


## Stage 2 (script extension): 5-anchor validation

Winners above plugged into the exact same 5-anchor (2018-2022) rollout mechanism as B-10/B-14, Tower 4
evaluation, 4-model ensemble (RF+XGB+LightGBM+SARIMAX) -- run as `b15_multi_anchor.py` (a single
5-anchor sweep takes ~2-3 minutes, same precedent as every B-09-B14 multi-anchor script).

**Result (5-anchor n-weighted mean, per-anchor-then-mean -- this project's established aggregation,
verified to reproduce D-54's exact published B-10 baseline figures):**

| Model | R2 | MASE |
|---|---|---|
| LightGBM_tuned | **0.017** | 0.950 |
| Ensemble_4model_tuned | 0.007 | 0.983 |
| XGB_tuned | -0.009 | 0.995 |
| SARIMAX | -0.054 | 1.038 |
| RF_tuned | -0.078 | 1.041 |

LightGBM_tuned is the best single model found across the entire B-09-B15 sequence, ahead of even
B-10's own ensemble (R2=0.012/0.0116 depending on rounding). Chain plots: `figures/b15_chains/
T4_anchor*.png` (via `b15_chain_plots.py`).

## Stage 3 (script extension): 3-way comparison vs B-10/B-14

`compile_b15_results.py` loads real per-bin data from all three sources (B-10's
`b10_ensemble_multi_anchor.csv`, B-14's `b14_tuned_rollout_summary.csv`, B-15's own
`b15_tuned_rollout_summary.csv` above), aligns by normalized model family (raw names differ across
sources, e.g. B-10's `SARIMAX` vs B-14's `SARIMAX_widened`), and computes the B-14-vs-B-15 win/loss/tie
verdict programmatically.

**Result:** no uniform winner between CV-tuning (B-14) and rollout-tuning (B-15) -- 2 families each,
1 tie. B-10's ensemble (R2=0.012) remains the best-validated **ensemble**; B-15's LightGBM is the best
single **model**. Full comparison table and narrative: `b15_results.md`.

## Addendum (script extensions): does tuning generalize across towers?

B-14/B-15's tuning above is Tower-4-only throughout (training pools T2+T4+T9, scoring never left T4).
`b15_cross_tower_eval.py` reuses the exact same pooled-fit models (no retraining) rolled out for T2/T9
targets too; `b15_t9_rollout_grid_search.py`/`b15_t9_multi_anchor.py` repeat Stage 1/2 above but scored
on Tower 9 instead (Tower 9 has usable real data at 4/5 anchors; Tower 2 only 1/5, too scarce to tune
independently).

**Headline finding:** LightGBM_tuned -- Tower 4's best model above -- is Tower 9's **worst**
(R2=-0.388 vs Tower 9's own best, the ensemble at -0.226). An independent tuning search scored on
Tower 9 picks genuinely different hyperparameters but barely changes the outcome (within +/-0.03 R2
of just reusing Tower 4's config) -- **Tower 9's poor performance is not primarily a tuning problem**.
Tower 2 (1 usable anchor) has too little real data for any reliable conclusion.

Full addendum narrative, tables, and chain-plot references: `b15_results.md`.

## Recommendation

- **Tower 4 (production):** B-10's unweighted 4-model ensemble (D-54, R2=0.012) remains the
  best-validated configuration. B-15's rollout-tuned LightGBM alone (R2=0.017) is the best single
  model found in the whole B-09-B15 sequence -- a rebalanced ensemble weighted toward it is a flagged,
  unexecuted follow-up.
- **Tower 9:** tuning does not help; reuse either config, the 4-model ensemble is least-bad (R2~-0.23).
- **Tower 2:** insufficient real evaluation data for any reliable conclusion.

**This closes the B-14/B-15 hyperparameter-tuning side-thread.** Full write-up: `b15_results.md`.
Cross-ref: D-54 (B-10 baseline), D-58 (B-14, the CV-tuned predecessor), D-59 (this experiment), D-53
(the anchor-count-caution lesson), D-57 (B-13's own Tower-2 scarcity finding, reconfirmed here).